### Jupyter를 이용하여 ROS 돌려보기

#### 추천 extensions 설치

![extension1](../imgs/extensions_img/extensions1.png)

![extension1](../imgs/extensions_img/extensions2.png)

![extension1](../imgs/extensions_img/extensions3.png)

![extension1](../imgs/extensions_img/extensions4.png)

![extension1](../imgs/extensions_img/extensions5.png)

를 설치하고

VSCode에서 ctrl + shift + p를 누른 후 

Preferences: Open User Settings (JSON) 을 선택하고

아래 내용을 추가하여 수정하고 저장한다. (이미 있다면 생략.)

In [ ]:
    "files.associations": {
        "*.repos": "yaml",
        "*.world": "xml",
        "*.xacro": "xml"
    },

    "colcon.provideTasks": true,
    "ros.distro": "humble"

저장 후 VSCode 재시작.

먼저 VSCode 터미널에 turtlesim 실행

```bash
ros2 run turtlesim turtlsim_node
```

In [ ]:
import rclpy as rp
from turtlesim.msg import Pose

rp.init()
test_node = rp.create_node('sub_test')

위 셀을 실행한 후 터미널에 아래 명령어를 통해 /sub_test 노드가 생성되었는지 확인한다.

```bash
ros2 node list
```

주의할 점은 위 셀은 딱 한번만 실행해야 하며, 두번 이상 실행할 경우 에러가 난다.

뒤에서도 나오겠지만 노드를 destroy해야 다시 실행할 수 있다.

Callback 함수 작성

In [3]:
def callback(data):
    print('--->')
    print('/turtle/pose : ', data)
    print('X : ', data.x)
    print('Y : ', data.y)
    print('Theta : ', data.theta)

#### subscriber 만들기

In [4]:
test_node.create_subscription(Pose, '/turtle1/pose', callback, 10)

test_node 구독

In [5]:
rp.spin_once(test_node)

--->
/turtle/pose :  turtlesim.msg.Pose(x=5.544444561004639, y=5.544444561004639, theta=0.0, linear_velocity=0.0, angular_velocity=0.0)
X :  5.544444561004639
Y :  5.544444561004639
Theta :  0.0


파이썬에서 노드를 돌리기 위해 spin()이라는 무한루프를 사용했다면

주피터에서는 이것 대신 spin_once()를 사용하여 노드를 딱 한번만 실행해준다.

주피터에서 무한루프인 spin()을 실행하게 되면 셀이 끝나지 않아 다른 셀을 실행할 수 없게 되기 때문이다.

In [7]:
rp.shutdown()

rcply를 이렇게 종료를 해 주어야

다른 셀에서 rclpy를 시작해 줄 수 있다.

#### Publisher 만들기

In [8]:
import rclpy as rp
from geometry_msgs.msg import Twist

rp.init()
test_node = rp.create_node('publish_test')

In [9]:
msg = Twist()
print(msg)

geometry_msgs.msg.Twist(linear=geometry_msgs.msg.Vector3(x=0.0, y=0.0, z=0.0), angular=geometry_msgs.msg.Vector3(x=0.0, y=0.0, z=0.0))


In [10]:
msg.linear.x = 2.0
pub = test_node.create_publisher(Twist, '/turtle1/cmd_vel', 10)
pub.publish(msg)

In [11]:
msg.linear.x = 1.0
msg.angular.z = 2.0
pub.publish(msg)

Timer를 이용해 토픽 발행하기

타이머 콜백 함수가 일정 횟수 이상 호출되지 않도록 조건문을 넣어줌.

In [12]:
cnt = 0

def timer_callback():
    global cnt
    cnt += 1
    print(cnt)
    pub.publish(msg)
    
    if cnt > 5 : 
        raise Exception('Publisher Stop')

In [13]:
timer_period = 2.0
timer = test_node.create_timer(timer_period, timer_callback)
rp.spin(test_node)

1
2
3
4
5
6


Exception: Publisher Stop

여기서는 횟수 제한을 두었기 때문에 spin을 써도 상관이 없다.

(횟수가 끝나면 종료되기 때문에.)

In [14]:
test_node.destroy_node()

노드 종료.

In [15]:
rp.shutdown()

서비스, 액션도 비슷하게 할 수 있음.

자세한 내역은 jupyter_ws 참고.

[jupyter_ws](../jupyter_ws)